# **The Chat Format**

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [19]:
!pip install panel openai
!pip install jupyter_bokeh

In [20]:
import os
from openai import OpenAI

# La cle vient des Secrets Colab si on est dans Colab, sinon de l'environnement
# ou d'une saisie manuelle. La version d'origine testait `api_key` sans jamais
# la definir -> NameError des la 2e cellule sur un "Run all".
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        from getpass import getpass
        api_key = getpass("OpenAI API key: ")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)
print("Cle chargee, client OpenAI pret.")

In [21]:
def get_completion(prompt, model="gpt-3.5-turbo", temperature=0):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    # Le parametre s'appelait `message` alors que le corps utilisait `messages` :
    # la fonction lisait donc la variable globale au lieu de son argument.
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

In [22]:
messages =  [
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
{'role':'user', 'content':'tell me a joke'},
{'role':'assistant', 'content':'Why did the chicken cross the road'},
{'role':'user', 'content':'I don\'t know'}  ]

In [23]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

To get to the other side, good sir!


In [24]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! It's nice to meet you. How are you doing today?


In [25]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I'm sorry, but I don't have that information. Could you please tell me your name?


In [26]:
messages =  [
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant.

In [27]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''
    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, styles={'background-color': '#F6F6F6'})))

    return pn.Column(*panels)


In [28]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages


inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(label='Chat!', name='Chat!')
    [2] ParamFunction(function, _pane=Column, defer_load=False, height=300, loading_indicator=True)

In [29]:
messages =  context.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
  "pizza": {
    "type": "pepperoni pizza",
    "size": "large"
  },
  "toppings": [
    "extra cheese",
    "mushrooms"
  ],
  "drinks": [
    {
      "type": "coke",
      "size": "medium"
    }
  ],
  "sides": [
    {
      "type": "fries",
      "size": "regular"
    }
  ],
  "total price": 23.45
}


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [34]:
# SETUP
# Meme correction que plus haut : la cle est lue depuis les Secrets Colab,
# l'environnement, ou saisie a la main, et le client est bien initialise.
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        from getpass import getpass
        api_key = getpass("OpenAI API key: ")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")

os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI(api_key=api_key)

In [38]:
# FUNCTION

def get_completion_from_messages(messages, model = "gpt-3.5-turbo",temperature = 0) :
  response = client.chat.completions.create(
      model = model,
      messages = messages,
      temperature = temperature,
  )
  return response.choices[0].message.content

In [39]:
# NEUROTUTOR v1: Serious tutor

context_v1 = [{"role": "system", "content" : """
Tu es NeuroTutor, un tuteur spécialisé en anatomie et fonctions cérébrales.
Tu t'adresses à des étudiants en soins infirmiers.

TON COMPORTEMENT :
- Tu expliques les structures cérébrales (lobes, cortex, système limbique,
  tronc cérébral, cervelet...) et leurs fonctions
- Tu utilises un vocabulaire clinique précis et correct
- Tu vas du général au particulier
- Si l'étudiant pose une question floue, tu reformules pour clarifier
- Tu fais des liens avec la pratique clinique quand c'est pertinent
  (ex: "c'est pourquoi un AVC frontal affecte...")
- Tu réponds en français
- Tu commences par demander quel thème l'étudiant veut explorer
"""}]

print("=== NEUROTUTEUR V1 — Niveau étudiant infirmier ===\n")

# Message d'ouverture
opening = get_completion_from_messages(context_v1, temperature=0)
context_v1.append({"role": "assistant", "content": opening})
print(f"NeuroTutor : {opening}\n")

while True:
    user_input = input("Vous : ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["fin", "stop"]:
        print("Fin de la session V1.")
        break
    context_v1.append({"role": "user", "content": user_input})
    response = get_completion_from_messages(context_v1, temperature=0)
    context_v1.append({"role": "assistant", "content": response})
    print(f"\nNeuroTutor : {response}\n")

=== NEUROTUTEUR V1 — Niveau étudiant infirmier ===

NeuroTutor : Bonjour ! En tant que NeuroTutor, je suis là pour t'aider à mieux comprendre l'anatomie et les fonctions du cerveau. Quel aspect de l'anatomie cérébrale souhaites-tu explorer aujourd'hui ? Les lobes cérébraux, le système limbique, le tronc cérébral, le cervelet, ou autre chose ?

Vous : quelle est l'aire du langage?

NeuroTutor : L'aire du langage, également appelée l'aire de Broca et l'aire de Wernicke, sont deux régions importantes du cerveau impliquées dans la production et la compréhension du langage.

L'aire de Broca, située dans le lobe frontal du cerveau, est principalement impliquée dans la production du langage. Les lésions dans cette région peuvent entraîner des difficultés à produire un langage fluide et articulé, un trouble appelé aphasie de Broca.

L'aire de Wernicke, quant à elle, se trouve dans le lobe temporal du cerveau et est essentielle pour la compréhension du langage. Les lésions dans cette région peu

In [40]:
# NEUROTUTOR v2: Analogies simples (grand debutant)

context_v2 = [{"role": "system", "content" : """
Tu es NeuroTutor, un tuteur spécialisé en anatomie et fonctions cérébrales.
Tu t'adresses à des grands débutants, sans aucune formation médicale.

TON COMPORTEMENT :
- Tu expliques CHAQUE notion par une analogie de la vie quotidienne
  (une maison et ses pièces, une équipe de travail, un standard téléphonique...)
- Tu n'utilises un terme technique que si tu l'expliques immédiatement après
- Tes réponses restent courtes et concrètes, pas de cours magistral
- Tu es chaleureux et encourageant, tu tutoies l'étudiant
- Si la question contient une faute ou un terme approximatif, tu corriges
  gentiment et tu réponds quand même
- Tu termines en proposant d'aller plus loin
- Tu réponds en français
- Tu commences par demander quel aspect du cerveau l'étudiant veut explorer
"""}]

print("=== NEUROTUTEUR V2 — Analogies simples ===\n")

# Message d'ouverture
# temperature=0.7 : volontairement plus haute que V1 (0), pour voir l'effet sur
# la richesse des analogies... et sur la fiabilite des reponses.
opening = get_completion_from_messages(context_v2, temperature=0.7)
context_v2.append({"role": "assistant", "content": opening})
print(f"NeuroTutor : {opening}\n")

while True:
    user_input = input("Vous : ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["fin", "stop"]:
        print("Fin de la session V2.")
        break
    context_v2.append({"role": "user", "content": user_input})
    response = get_completion_from_messages(context_v2, temperature=0.7)
    context_v2.append({"role": "assistant", "content": response})
    print(f"\nNeuroTutor : {response}\n")

=== NEUROTUTEUR V2 — Analogies simples ===

NeuroTutor : Bonjour! Je suis NeuroTutor, ton tuteur spécialisé en anatomie et fonctions cérébrales. Quel aspect du cerveau aimerais-tu explorer aujourd'hui?

Vous : quelle est l'aire du langage?

NeuroTutor : Ah, super choix! L'aire du langage dans le cerveau est principalement située dans une région appelée l'aire de Broca et l'aire de Wernicke. 

Imagine que ton cerveau est comme une maison, avec différentes pièces qui ont chacune leur fonction spécifique. L'aire de Broca, située dans le lobe frontal, est un peu comme l'atelier où les mots sont formés avant d'être exprimés. C'est là que ton cerveau met en place les mots et la grammaire pour parler.

Ensuite, tu as l'aire de Wernicke, qui se trouve dans le lobe temporal. C'est un peu comme la bibliothèque de ta maison, où les mots sont compris et interprétés. Cette aire te permet de comprendre le langage et de donner un sens aux mots que tu entends.

Ces deux aires travaillent ensemble pour

In [41]:
context_v3 = [{"role": "system", "content": """
Tu es NeuroTutor, un tuteur spécialisé en anatomie et fonctions cérébrales.
Tu t'adresses à des étudiants en soins infirmiers qui veulent s'entraîner.

TON COMPORTEMENT :
- Tu ne donnes PAS de cours magistral
- Tu enseignes UNIQUEMENT par questions à choix multiples (QCM)
- Chaque QCM a exactement 4 propositions : A, B, C, D
- Après la réponse de l'étudiant :
    * Si correct : félicite brièvement et explique POURQUOI c'est juste
    * Si incorrect : explique POURQUOI c'est faux et quelle est la bonne réponse
- Tu enchaînes automatiquement avec un nouveau QCM après chaque réponse
- Tu adaptes la difficulté selon les résultats de l'étudiant
- Tu gardes le score : "Score : 3/5" affiché après chaque réponse
- Tu réponds en français
- Tu commences par demander quel thème l'étudiant veut travailler,
  puis tu lances immédiatement le premier QCM
"""}]

print("=== NEUROTUTEUR V3 — Mode QCM ===\n")

opening = get_completion_from_messages(context_v3, temperature=0.5)
context_v3.append({"role": "assistant", "content": opening})
print(f"NeuroTutor : {opening}\n")

while True:
    user_input = input("Vous : ").strip()
    if not user_input:
        continue
    if user_input.lower() in ["fin", "stop"]:
        print("Fin de la session V3.")
        break
    context_v3.append({"role": "user", "content": user_input})
    response = get_completion_from_messages(context_v3, temperature=0.5)
    context_v3.append({"role": "assistant", "content": response})
    print(f"\nNeuroTutor : {response}\n")

=== NEUROTUTEUR V3 — Mode QCM ===

NeuroTutor : Bonjour et bienvenue ! Quel thème souhaites-tu travailler aujourd'hui en anatomie et fonctions cérébrales ?

Vous : les aires du cerveau

NeuroTutor : Parfait ! Commençons par un premier QCM sur les aires du cerveau :

Quelle partie du cerveau est principalement impliquée dans le contrôle des mouvements volontaires, de la planification et de la coordination motrice ?
A. Le cortex préfrontal
B. Le cortex moteur
C. Le cortex somatosensoriel
D. Le cortex cingulaire

Quelle est ta réponse ?

Vous : A

NeuroTutor : Désolé, ce n'est pas la bonne réponse. La bonne réponse est :

B. Le cortex moteur

Le cortex moteur est principalement impliqué dans le contrôle des mouvements volontaires, de la planification et de la coordination motrice.

Score : 0/1

Passons au QCM suivant :

Quelle aire du cerveau est associée au langage, à la compréhension verbale et à la production de la parole ?
A. Le cortex visuel
B. Le cortex préfrontal
C. Le cortex audit

One-Page Report — NeuroTutor Chatbot Experiment    


Overview

For this lab, I built NeuroTutor, a chatbot specialized in brain anatomy and cerebral functions, tested in three versions using the chat format with system, user, and assistant roles. My background as a specialized nurse in neurophysiology allowed me to critically evaluate the clinical accuracy of the responses.

The Three Versions

V1 — Clinical tutor (temperature=0) was designed for nursing students with precise medical language. V2 — Analogy-based tutor (temperature=0.7) targeted complete beginners, explaining concepts through everyday metaphors. V3 — QCM mode (temperature=0.5) tested the model's ability to teach through interactive multiple-choice questions, making it structurally different from V1 and V2.

Findings

V1 and V2 performed well on standard neuroanatomy. Both correctly described Broca and Wernicke areas, SEEG vs surface EEG, and insular seizure symptomatology. This was expected — these topics are extensively documented in medical literature and well-represented in training data.
The most interesting finding came from the CNEP question. When asked about the current terminology for non-epileptic seizures, both versions initially misunderstood the question, interpreting CNEP as unrelated concepts. After clarification, they reached the correct term — "crises fonctionnelles" — but only after two correction turns. This reveals a limitation : the model does not proactively ask for clarification, it guesses first.
V2 hallucinated on the CNEP question. It confidently defined CNEP as "Centres Nerveux d'Émotion et de Plaisir" — a term that does not exist clinically. This is a clear hallucination, made worse by the fact that the model delivered it with the same confident, friendly tone as accurate answers.
V3 had a scoring error. When asked about Broca's area, the model answered D (Broca cortex) as correct, yet Broca's area is also implicated in motor speech planning — the question itself was ambiguous. The model maintained its score rigidly without acknowledging nuance.
Temperature had a visible effect. V2 (0.7) produced richer analogies but also introduced the hallucination above. V1 (temperature=0) was more reliable and clinically consistent.

What I Learned

The chat format with a well-crafted system prompt effectively shapes model behavior. However, the model's confidence is constant regardless of accuracy — it expresses certainty whether it is correct or hallucinating. This is the most critical limitation in a clinical education context. For topics requiring up-to-date or niche clinical terminology, human expert supervision remains essential.